In [23]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

In [24]:
#=========
# Setup
#=========
df = pd.DataFrame({
    "order_id": [101 , 102 , 103 , 104 , 105 , 106 , 107 , 108] ,
    "customer_id": [1 , 1 , 2 , 3 , 4 , 4 , 5 , 6] ,
    "region": ["East" , "East" , "West" , "West" , "East" , "North" , "West" , "East"] ,
    "product": ["Cable" , "Wire" , "Cable" , "Wire" , "Cable" , "Wire" , "Wire" , "Cable"] ,
    "month": pd.to_datetime(["2025-11" , "2025-11" , "2025-11" , "2025-12" , "2025-12" , "2025-12" , "2025-12" , "2025-11"]) ,
    "sales": [120 , 80 , 90 , 110 , 150 , 70 , 60 , 200] ,
    "returns": [5 , 2 , 3 , 6 , 4 , 1 , 7 , np.nan] ,
})
df["return_rate"] = df["returns"] / df["sales"]
df

,order_id,customer_id,region,product,month,sales,returns,return_rate
0,101,1,East,Cable,2025-11-01,120,5.0,0.041667
1,102,1,East,Wire,2025-11-01,80,2.0,0.025000
2,103,2,West,Cable,2025-11-01,90,3.0,0.033333
3,104,3,West,Wire,2025-12-01,110,6.0,0.054545
4,105,4,East,Cable,2025-12-01,150,4.0,0.026667
5,106,4,North,Wire,2025-12-01,70,1.0,0.014286
6,107,5,West,Wire,2025-12-01,60,7.0,0.116667
7,108,6,East,Cable,2025-11-01,200,NaN,NaN


In [25]:
#============================================
# Case 1) Sanity check: info, dtypes, shape
#============================================
df.shape
print("\n")
df.dtypes
print("\n")
df.info()

(8, 8)

order_id                int64
customer_id             int64
region                 object
product                object
month          datetime64[ns]
sales                   int64
returns               float64
return_rate           float64
dtype: object



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     8 non-null      int64         
 1   customer_id  8 non-null      int64         
 2   region       8 non-null      object        
 3   product      8 non-null      object        
 4   month        8 non-null      datetime64[ns]
 5   sales        8 non-null      int64         
 6   returns      7 non-null      float64       
 7   return_rate  7 non-null      float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(2)
memory usage: 644.0+ bytes


In [26]:
#=================================
# Case 2) describe + percentiles
#=================================
df[["sales" , "returns" , "return_rate"]].describe(percentiles = [0.05 , 0.5 , 0.95]).round(3)

,sales,returns,return_rate
count,8.000,7.00,7.000
mean,110.000,4.00,0.045
std,46.599,2.16,0.034
min,60.000,1.00,0.014
5%,63.500,1.30,0.018
50%,100.000,4.00,0.033
95%,182.500,6.70,0.098
max,200.000,7.00,0.117


In [27]:
#=================================
# Case 3) value_counts + nunique
#=================================
df["region"].nunique(dropna = False)
print("\n")
df["region"].value_counts(normalize = True).round(3)
print("\n")
df["product"].value_counts()

3

region
East     0.500
West     0.375
North    0.125
Name: proportion, dtype: float64

product
Cable    4
Wire     4
Name: count, dtype: int64

In [28]:
#=========================
# Case 4) groupby + agg
#=========================
kpi = (
    df.groupby(["region" , "product"] , as_index = False)
        .agg(
            orders = ("order_id" , "count") , total_sales = ("sales" , "sum") ,
            avg_sales = ("sales" , "mean") , total_returns = ("returns" , "sum") ,
            avg_return_rate = ("return_rate" , "mean") ,
        ).sort_values(["total_sales"] , ascending = False)
)
kpi.round(2)

,region,product,orders,total_sales,avg_sales,total_returns,avg_return_rate
0,East,Cable,3,470,156.67,9.0,0.03
4,West,Wire,2,170,85.00,13.0,0.09
3,West,Cable,1,90,90.00,3.0,0.03
1,East,Wire,1,80,80.00,2.0,0.02
2,North,Wire,1,70,70.00,1.0,0.01


In [29]:
#======================================
# Case 5) corr/cov + groupby().cov()
#======================================
num = df[["sales" , "returns"]].copy()
num.corr(numeric_only = True).round(2)
print("\n")
num.cov(numeric_only=True).round(2)
cols = ["sales" , "returns"]
g_cov = (
    df.dropna(subset = cols).groupby("region").filter(lambda g: len(g) >= 2)
        .groupby("region")[cols].cov()
)
print("\n")
g_cov.round(2)

,sales,returns
sales,1.00,0.15
returns,0.15,1.00


,sales,returns
sales,2171.43,10.00
returns,10.00,4.67


sales  returns
region                          
East   sales    1233.33    38.33
       returns    38.33     2.33
West   sales     633.33   -18.33
       returns   -18.33     4.33

In [30]:
#===============================
# Case 6) extremes + crosstab
#===============================
df.nlargest(3 , "sales")[["order_id" , "region" , "product" , "sales"]]
print("\n")
df.nsmallest(3 , "sales")[["order_id" , "region" , "product" , "sales"]]
mix = pd.crosstab(df["region"] , df["product"] , normalize = "index").round(2)
print("\n")
mix

,order_id,region,product,sales
7,108,East,Cable,200
4,105,East,Cable,150
0,101,East,Cable,120


,order_id,region,product,sales
6,107,West,Wire,60
5,106,North,Wire,70
1,102,East,Wire,80


product,Cable,Wire
region,,
East,0.75,0.25
North,0.00,1.00
West,0.33,0.67
